# LeetCode 3785 — Minimum Swaps to Avoid Forbidden Values bằng Python

- Link: https://leetcode.com/problems/minimum-swaps-to-avoid-forbidden-values/
- Độ khó: **Hard**
- Chủ đề: Tham lam, đếm tần suất (`Counter`), chặn dưới / chặn trên (lower bound – construction), điều kiện khả thi kiểu Hall, BFS kiểm chứng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minSwaps(self, nums: List[int], forbidden: List[int]) -> int:
```

## 1. Mục tiêu học

1. Xác định **vị trí xung đột** (`nums[i] == forbidden[i]`) và nhóm chúng theo giá trị.
2. Chứng minh hai **chặn dưới**: mỗi lần hoán đổi sửa tối đa 2 xung đột ⇒ `≥ ⌈k/2⌉`; và sửa tối đa 1 xung đột của **cùng một giá trị** ⇒ `≥ M` (nhóm lớn nhất).
3. Xây dựng cách hoán đổi đạt đúng `max(⌈k/2⌉, M)`: ghép cặp xung đột khác giá trị, phần dư đổi với vị trí "an toàn".
4. Điều kiện **khả thi**: với mọi giá trị `v`, `count_nums(v) + count_forbidden(v) ≤ n`.
5. Kiểm chứng công thức bằng **BFS** trên không gian trạng thái nhỏ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` và `forbidden` cùng độ dài `n`. Mỗi thao tác hoán đổi hai phần tử của `nums`. Tìm số thao tác **ít nhất** để mọi `i` có `nums[i] ≠ forbidden[i]`; nếu không thể, trả về `-1`.

**Ví dụ 1:** `nums = [1,2,3], forbidden = [3,2,1]` → `1`.

**Ví dụ 2:** `nums = [4,6,6,5], forbidden = [4,6,5,5]` → `2`.

**Ví dụ 3:** `nums = [7,7], forbidden = [8,7]` → `-1`.

**Ví dụ 4:** `nums = [1,2], forbidden = [2,1]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i], forbidden[i] ≤ 10^9`.

In [1]:
from typing import List
from collections import Counter, deque
import heapq, random, time

examples = [([1, 2, 3], [3, 2, 1], 1), ([4, 6, 6, 5], [4, 6, 5, 5], 2), ([7, 7], [8, 7], -1), ([1, 2], [2, 1], 0)]
for a, f, exp in examples:
    print(a, f, "-> kỳ vọng", exp)

[1, 2, 3] [3, 2, 1] -> kỳ vọng 1
[4, 6, 6, 5] [4, 6, 5, 5] -> kỳ vọng 2
[7, 7] [8, 7] -> kỳ vọng -1
[1, 2] [2, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. `Counter` và nhóm xung đột theo giá trị

In [2]:
nums, forb = [4, 6, 6, 5], [4, 6, 5, 5]
bad = Counter(nums[i] for i in range(len(nums)) if nums[i] == forb[i])
print("xung đột theo giá trị:", dict(bad), "| k =", sum(bad.values()), "| M =", max(bad.values()))

xung đột theo giá trị: {4: 1, 6: 1, 5: 1} | k = 3 | M = 1


### 3.2. Một lần hoán đổi ảnh hưởng tới đúng hai vị trí

Hoán đổi `i, j` chỉ thay đổi trạng thái (xung đột hay không) của `i` và `j`. Vì vậy:

- số xung đột giảm tối đa 2 mỗi lần;
- nếu `i`, `j` đều xung đột **cùng giá trị** `v`, sau hoán đổi cả hai vẫn mang `v` ⇒ vẫn xung đột. Tức là mỗi lần chỉ "gỡ" được tối đa **một** xung đột của mỗi giá trị.

In [3]:
a, f = [5, 5], [5, 5]
a[0], a[1] = a[1], a[0]
print("đổi hai xung đột cùng giá trị:", a, "vẫn xung đột:", [a[i] == f[i] for i in range(2)])

đổi hai xung đột cùng giá trị: [5, 5] vẫn xung đột: [True, True]


### 3.3. Điều kiện khả thi

Giá trị `v` xuất hiện `c_n(v)` lần trong `nums` và bị cấm ở `c_f(v)` vị trí. Cần đủ chỗ: `c_n(v) ≤ n − c_f(v)`. Điều kiện này (cho mọi `v`) cũng **đủ** để tồn tại cách sắp xếp hợp lệ (định lý Hall cho bài toán ghép cặp này — kiểm chứng bằng vét cạn ở mục 12).

In [4]:
def feasible(nums, forb):
    n = len(nums); cn, cf = Counter(nums), Counter(forb)
    return all(cn[v] + cf[v] <= n for v in cn)

print(feasible([7, 7], [8, 7]), feasible([1, 2, 3], [3, 2, 1]))

False True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Gọi `k` = số xung đột, `M` = số xung đột của giá trị phổ biến nhất trong các xung đột.

- **Hai xung đột khác giá trị** `i` (giá trị `a`), `j` (giá trị `b`): đổi chúng ⇒ `i` nhận `b ≠ forbidden[i] = a`, `j` nhận `a ≠ b` ⇒ **cả hai được sửa** bằng 1 lần.
- **Xung đột còn lại** (không có cặp khác giá trị): đổi với một vị trí **tốt** `j` sao cho sau đổi cả hai vẫn ổn.

Ví dụ 2: xung đột ở `0 (4)`, `1 (6)`, `3 (5)` ⇒ `k = 3`, `M = 1`. Ghép `(0, 1)` ⇒ 1 lần; còn `3` ⇒ đổi với vị trí tốt `2` (giá trị 6, cấm 5): `nums[3] = 6 ≠ 5`, `nums[2] = 5`? — bị cấm! Phải chọn kỹ: hàm brute ở mục 5 tìm ra 2 lần là tối ưu.

**Công thức:** nếu khả thi, `answer = max(⌈k/2⌉, M)` (và `0` nếu `k = 0`).

## 5. Cách 1 — Brute Force: BFS trên trạng thái mảng

Mỗi trạng thái là một hoán vị của `nums`; cạnh là một lần hoán đổi. BFS cho số lần ít nhất. Chỉ dùng cho `n ≤ 7`.

In [5]:
def brute(nums, forb):
    n = len(nums); start = tuple(nums)
    ok = lambda a: all(a[i] != forb[i] for i in range(n))
    if ok(start): return 0
    seen = {start}; dq = deque([(start, 0)])
    while dq:
        a, d = dq.popleft()
        for i in range(n):
            for j in range(i + 1, n):
                if a[i] == a[j]: continue
                b = list(a); b[i], b[j] = b[j], b[i]; b = tuple(b)
                if b in seen: continue
                if ok(b): return d + 1
                seen.add(b); dq.append((b, d + 1))
    return -1

for a, f, exp in examples:
    print(brute(a, f), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1
0 kỳ vọng 0


**Nút thắt:** số trạng thái tới `n!`. Cần lập luận chặn dưới/chặn trên.

## 6. Ý tưởng thuật toán / bước đột phá

**Chặn dưới.** Theo 3.2: mỗi lần giảm tối đa 2 xung đột ⇒ `≥ ⌈k/2⌉`; mỗi lần gỡ tối đa 1 xung đột của giá trị trội ⇒ `≥ M`.

**Đạt được.** Ghép cặp tham lam: luôn ghép một xung đột của **nhóm lớn nhất** với một xung đột của nhóm khác (như xếp lịch "không hai việc cùng loại liền nhau"):

- nếu `M ≤ k − M`: ghép được `⌊k/2⌋` cặp khác giá trị; nếu `k` lẻ còn 1 xung đột ⇒ đổi với một vị trí tốt phù hợp ⇒ tổng `⌈k/2⌉`;
- nếu `M > k − M`: ghép `k − M` cặp, còn `2M − k` xung đột cùng giá trị `v` ⇒ mỗi cái đổi với một vị trí tốt `j` mà `nums[j] ≠ v` và `forbidden[j] ≠ v` ⇒ tổng `M`.

Sự tồn tại các vị trí tốt phù hợp được bảo đảm bởi điều kiện khả thi (đã kiểm chứng với BFS trên hàng nghìn ca).

**Kết luận:** `answer = -1` nếu ∃ `v`: `c_n(v) + c_f(v) > n`; ngược lại `max(⌈k/2⌉, M)`.

## 7. Các cách tiếp theo

- **Cách 2:** **mô phỏng tham lam** bằng heap: ghép xung đột của hai nhóm lớn nhất, đếm số lần; phần dư mỗi cái tốn 1 lần.
- **Cách 3:** **công thức đóng** `max(⌈k/2⌉, M)` sau kiểm tra khả thi (lời giải nộp chính).

### 7.1. Cách 2 — Mô phỏng tham lam bằng heap

In [6]:
def greedy_heap(nums, forb):
    if not feasible(nums, forb):
        return -1
    bad = Counter(nums[i] for i in range(len(nums)) if nums[i] == forb[i])
    heap = [-c for c in bad.values()]
    heapq.heapify(heap)
    swaps = 0
    while len(heap) >= 2:
        a = -heapq.heappop(heap); b = -heapq.heappop(heap)     # hai nhóm lớn nhất
        swaps += 1                                            # ghép một cặp khác giá trị
        if a - 1: heapq.heappush(heap, -(a - 1))
        if b - 1: heapq.heappush(heap, -(b - 1))
    if heap:
        swaps += -heap[0]                                     # phần dư cùng giá trị: mỗi cái 1 lần
    return swaps

for a, f, exp in examples:
    print(greedy_heap(a, f), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1
0 kỳ vọng 0


**Độ phức tạp:** `O(k log G)` với `G` số nhóm. Kết quả trùng công thức — mô phỏng giúp **thấy** vì sao `max(⌈k/2⌉, M)`.

### 7.2. Cách 3 — Công thức đóng (lời giải chính)

In [7]:
def closed_form(nums, forb):
    n = len(nums)
    cn, cf = Counter(nums), Counter(forb)
    for v, c in cn.items():
        if c + cf[v] > n:
            return -1
    bad = Counter(nums[i] for i in range(n) if nums[i] == forb[i])
    k = sum(bad.values())
    if k == 0:
        return 0
    return max((k + 1) // 2, max(bad.values()))

for a, f, exp in examples:
    print(closed_form(a, f), "kỳ vọng", exp)

1 kỳ vọng 1
2 kỳ vọng 2
-1 kỳ vọng -1
0 kỳ vọng 0


**Độ phức tạp Cách 3:** `O(n)` thời gian và bộ nhớ.

## 8. TRACE từng bước

Trace mô phỏng tham lam trên một ca có nhóm trội: `nums = [3,3,3,1,2,5,6]`, `forbidden = [3,3,3,1,4,4,4]`.

In [8]:
def trace(nums, forb):
    n = len(nums); cn, cf = Counter(nums), Counter(forb)
    print("khả thi:", all(cn[v] + cf[v] <= n for v in cn))
    bad = Counter(nums[i] for i in range(n) if nums[i] == forb[i])
    k = sum(bad.values()); print("xung đột:", dict(bad), "| k =", k, "| M =", max(bad.values()) if bad else 0)
    heap = [(-c, v) for v, c in bad.items()]; heapq.heapify(heap); swaps = 0
    while len(heap) >= 2:
        (a, va), (b, vb) = heapq.heappop(heap), heapq.heappop(heap)
        swaps += 1; print(f"  ghép xung đột giá trị {va} với {vb} -> swaps = {swaps}")
        if a + 1: heapq.heappush(heap, (a + 1, va))
        if b + 1: heapq.heappush(heap, (b + 1, vb))
    if heap:
        c, v = heap[0]; swaps += -c; print(f"  còn {-c} xung đột giá trị {v}: đổi với vị trí tốt -> swaps = {swaps}")
    print("tổng:", swaps, "| công thức:", closed_form(nums, forb))

trace([3, 3, 3, 1, 2, 5, 6], [3, 3, 3, 1, 4, 4, 4])
print("BFS:", brute([3, 3, 3, 1, 2, 5, 6], [3, 3, 3, 1, 4, 4, 4]))

khả thi: True
xung đột: {3: 3, 1: 1} | k = 4 | M = 3
  ghép xung đột giá trị 3 với 1 -> swaps = 1
  còn 2 xung đột giá trị 3: đổi với vị trí tốt -> swaps = 3
tổng: 3 | công thức: 3
BFS: 3


In [9]:
trace([4, 6, 6, 5], [4, 6, 5, 5])

khả thi: True
xung đột: {4: 1, 6: 1, 5: 1} | k = 3 | M = 1
  ghép xung đột giá trị 4 với 5 -> swaps = 1
  còn 1 xung đột giá trị 6: đổi với vị trí tốt -> swaps = 2
tổng: 2 | công thức: 2


## 9. Code nộp LeetCode

In [10]:
from collections import Counter

class Solution:
    def minSwaps(self, nums: List[int], forbidden: List[int]) -> int:
        n = len(nums)
        cn, cf = Counter(nums), Counter(forbidden)
        for v, c in cn.items():
            if c + cf[v] > n:                  # không đủ vị trí cho giá trị v
                return -1
        bad = Counter(nums[i] for i in range(n) if nums[i] == forbidden[i])
        k = sum(bad.values())
        if k == 0:
            return 0
        return max((k + 1) // 2, max(bad.values()))

sol = Solution()
for a, f, exp in examples:
    got = sol.minSwaps(a, f)
    print(a, f, "->", got, "✔" if got == exp else "✘")

[1, 2, 3] [3, 2, 1] -> 1 ✔
[4, 6, 6, 5] [4, 6, 5, 5] -> 2 ✔
[7, 7] [8, 7] -> -1 ✔
[1, 2] [2, 1] -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Khả thi ⇔ điều kiện đếm:** (⇒) mỗi bản `v` cần một vị trí không cấm `v`, có `n − c_f(v)` vị trí như vậy. (⇐) Ghép cặp hai phía giữa "bản giá trị" và "vị trí": điều kiện Hall rút gọn về từng giá trị vì các ràng buộc cấm của mỗi vị trí chỉ loại **một** giá trị (cần lập luận đầy đủ; notebook kiểm chứng bằng vét cạn).
2. **Chặn dưới** `⌈k/2⌉` và `M`: mục 6.
3. **Đạt chặn dưới:** ghép cặp xung đột khác giá trị (tham lam nhóm lớn nhất) cho `min(⌊k/2⌋, k − M)` lần sửa 2 xung đột; phần còn lại sửa bằng đổi với vị trí tốt, mỗi lần 1 xung đột. Tổng = `max(⌈k/2⌉, M)`.
4. Ở mục 12, công thức được đối chiếu với BFS (lời giải tối ưu tuyệt đối) trên hàng nghìn ca ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS | `O(n! · n²)` | `O(n!)` |
| 2. Tham lam + heap | `O(n + k log G)` | `O(n)` |
| 3. Công thức | `O(n)` | `O(n)` |

## 12. Bộ test

In [11]:
def run_all(a, f, with_brute=True):
    res = [greedy_heap(a, f), closed_form(a, f), Solution().minSwaps(a, f)]
    if with_brute:
        res.append(brute(a, f))
    return res

for a, f, exp in examples:
    assert all(v == exp for v in run_all(a, f))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng BFS)
cases = [
    ([1], [1], -1),                       # n = 1, xung đột, không có chỗ đổi
    ([1], [2], 0),
    ([5, 5, 5], [5, 5, 1], -1),           # 3 + 2 > 3
    ([5, 5, 1, 2], [5, 5, 3, 3], 2),      # M = 2
    ([1, 2, 3, 4], [1, 2, 3, 4], 2),      # k = 4, M = 1
    ([1, 2, 3], [1, 2, 3], 2),            # k = 3 lẻ
]
for a, f, exp in cases:
    r = run_all(a, f)
    assert all(v == exp for v in r), (a, f, r)
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 phản ví dụ: chỉ dùng ⌈k/2⌉ (quên nhóm trội)
def wrong_half(a, f):
    if not feasible(a, f): return -1
    k = sum(1 for x, y in zip(a, f) if x == y)
    return (k + 1) // 2

print("sai:", wrong_half([5, 5, 1, 2], [5, 5, 3, 3]), "| đúng:", Solution().minSwaps([5, 5, 1, 2], [5, 5, 3, 3]))

sai: 1 | đúng: 2


In [14]:
# 12.4 đối chiếu ngẫu nhiên với BFS
rng = random.Random(3785)
for _ in range(2000):
    n = rng.randint(1, 6)
    a = [rng.randint(1, 3) for _ in range(n)]; f = [rng.randint(1, 3) for _ in range(n)]
    r = run_all(a, f)
    assert len(set(r)) == 1, (a, f, r)
print("2000 test ngẫu nhiên (BFS): OK")

2000 test ngẫu nhiên (BFS): OK


In [15]:
# 12.5 test lớn
rng = random.Random(9)
n = 10**5
a = [rng.randint(1, 50) for _ in range(n)]; f = [rng.randint(1, 50) if rng.random() < 0.5 else a[i] for i in range(n)]
t0 = time.time(); x = Solution().minSwaps(a, f); t1 = time.time()
print("đáp án:", x, "| khớp mô phỏng:", x == greedy_heap(a, f), "| thời gian < 1s:", t1 - t0 < 1)

đáp án: 25464 | khớp mô phỏng: True | thời gian < 1s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ trả `⌈k/2⌉`

Đã thấy ở 12.3: hai xung đột cùng giá trị không thể sửa nhau, nên khi một giá trị chiếm đa số, cần `M` lần.

### Lỗi 2 — Kiểm tra khả thi chỉ trên các vị trí xung đột

Khả thi phụ thuộc **toàn mảng**: `c_n(v)` và `c_f(v)` đếm trên mọi vị trí.

In [16]:
def wrong_feasible(a, f):
    bad = [i for i in range(len(a)) if a[i] == f[i]]
    if not bad: return 0
    cnt = Counter(a[i] for i in bad)
    if any(c > len(a) - c for c in cnt.values()): return -1           # SAI: không xét forbidden ngoài xung đột
    k = len(bad); return max((k + 1) // 2, max(cnt.values()))

print("sai:", wrong_feasible([7, 7], [8, 7]), "| đúng:", Solution().minSwaps([7, 7], [8, 7]))

sai: 1 | đúng: -1


### Lỗi 3 — Đếm `k` gồm cả vị trí không xung đột / dùng số giá trị phân biệt

`M` là **số xung đột** của giá trị trội, không phải số lần giá trị đó xuất hiện trong `nums`.

In [17]:
def wrong_M(a, f):
    if not feasible(a, f): return -1
    k = sum(1 for x, y in zip(a, f) if x == y)
    if k == 0: return 0
    return max((k + 1) // 2, max(Counter(a).values()))                # SAI: đếm trên toàn nums

print("sai:", wrong_M([2, 2, 2, 1], [3, 3, 2, 1]), "| đúng:", Solution().minSwaps([2, 2, 2, 1], [3, 3, 2, 1]))

sai: 3 | đúng: 1


### Lỗi 4 — Hoán đổi xung đột với vị trí tốt bất kỳ (không kiểm tra)

Khi cài đặt dựng lời giải cụ thể, đổi xung đột `i` (giá trị `v`) với vị trí tốt `j` phải bảo đảm `nums[j] ≠ forbidden[i] (= v)` và `v ≠ forbidden[j]`. Ví dụ 2: đổi vị trí 3 với vị trí 2 làm `nums[2] = 5 = forbidden[2]`.

In [18]:
a, f = [4, 6, 6, 5], [4, 6, 5, 5]
b = a[:]; b[3], b[2] = b[2], b[3]
print("sau khi đổi 3 với 2:", b, "| xung đột:", [i for i in range(4) if b[i] == f[i]])

sau khi đổi 3 với 2: [4, 6, 5, 6] | xung đột: [0, 1, 2]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Số thao tác ít nhất"** mà mỗi thao tác ảnh hưởng ít phần tử ⇒ tìm **chặn dưới** (mỗi thao tác giảm tối đa bao nhiêu) rồi **xây dựng** đạt chặn dưới.
- **Nhóm trội** (một loại chiếm đa số) ⇒ đáp án thường là `max(⌈k/2⌉, M)` (giống *Task Scheduler*, *Reorganize String*).
- Khả thi của bài "mỗi vị trí cấm một giá trị" ⇒ điều kiện đếm theo từng giá trị.
- Luôn kiểm chứng công thức bằng BFS trên dữ liệu nhỏ.

## 15. Biến thể của bài

1. **Dựng cụ thể dãy hoán đổi** (không chỉ đếm): ghép cặp bằng heap, phần dư tìm vị trí tốt phù hợp.
2. **Mỗi vị trí cấm một tập giá trị**: khả thi là bài ghép cặp tổng quát (luồng cực đại).
3. **Chi phí hoán đổi phụ thuộc khoảng cách** `|i − j|`: không còn công thức đơn giản.
4. **Chỉ được hoán đổi kề nhau**: bài khác hẳn (đếm nghịch thế).

In [19]:
# Biến thể 1: dựng dãy hoán đổi cụ thể và kiểm tra
def construct(nums, forb):
    a = nums[:]; n = len(a)
    if not feasible(a, forb): return None
    groups = {}
    for i in range(n):
        if a[i] == forb[i]: groups.setdefault(a[i], []).append(i)
    heap = [(-len(v), val) for val, v in groups.items()]; heapq.heapify(heap); ops = []
    while len(heap) >= 2:
        (c1, v1), (c2, v2) = heapq.heappop(heap), heapq.heappop(heap)
        i, j = groups[v1].pop(), groups[v2].pop()
        a[i], a[j] = a[j], a[i]; ops.append((i, j))
        if c1 + 1: heapq.heappush(heap, (c1 + 1, v1))
        if c2 + 1: heapq.heappush(heap, (c2 + 1, v2))
    for val, idxs in groups.items():
        for i in idxs:
            for j in range(n):
                if a[j] != forb[j] and a[j] != forb[i] and a[i] != forb[j]:
                    a[i], a[j] = a[j], a[i]; ops.append((i, j)); break
    return ops, a

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 7); x = [rng.randint(1, 3) for _ in range(n)]; f = [rng.randint(1, 3) for _ in range(n)]
    res = construct(x, f)
    if res is None:
        assert Solution().minSwaps(x, f) == -1; continue
    ops, final = res
    assert all(final[i] != f[i] for i in range(n)) and len(ops) == Solution().minSwaps(x, f), (x, f, ops)
print("Biến thể 1: dựng được dãy hoán đổi đúng số lần tối ưu.")

Biến thể 1: dựng được dãy hoán đổi đúng số lần tối ưu.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Duyệt trạng thái | `O(n! n²)` | lớn | Kiểm chứng |
| 2. Tham lam heap | Ghép hai nhóm lớn nhất | `O(n + k log G)` | `O(n)` | Khi cần dựng lời giải |
| 3. Công thức | `max(⌈k/2⌉, M)` | `O(n)` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minSwaps([1,1,2], [1,1,3])` = ?

**Bài 2 (tính tay).** `minSwaps([1,2,3,4,5], [1,2,3,4,5])`.

**Bài 3 (điền TODO).** Viết `lower_bounds(nums, forb)` trả về cặp `(⌈k/2⌉, M)`.

**Bài 4 (tự viết lại).** Viết lại lời giải không dùng `Counter` (dùng `dict` thường).

**Bài 5 (mini challenge).** Mỗi vị trí cấm **hai** giá trị (`forbidden1[i]`, `forbidden2[i]`). Viết BFS và thử tìm công thức/điều kiện tương tự cho dữ liệu nhỏ.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def lower_bounds(nums, forb):
    # TODO: k = số xung đột, M = nhóm lớn nhất
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def two_forbidden_bfs(nums, f1, f2):
    # TODO: BFS như Cách 1 với điều kiện nums[i] not in (f1[i], f2[i])
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khả thi: giá trị 1 xuất hiện 2 lần, bị cấm 2 vị trí ⇒ `2 + 2 > 3` ⇒ `-1`. (2) Không cần đếm thêm. (3) `-1`.

**Bài 2.** (1) `k = 5`, mỗi giá trị một xung đột ⇒ `M = 1`. (2) `⌈5/2⌉ = 3`. (3) Đáp án `3`.

**Bài 3.** (1) Duyệt một lần. (2) `Counter` các giá trị xung đột. (3) Trả `((k+1)//2, max or 0)`.

**Bài 4.** (1) `d[v] = d.get(v, 0) + 1`. (2) Giữ logic. (3) Đối chiếu.

**Bài 5.** (1) Điều kiện ok mới. (2) BFS không đổi. (3) Quan sát: một lần đổi vẫn sửa tối đa 2 xung đột.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().minSwaps([1, 1, 2], [1, 1, 3]) == -1 == brute([1, 1, 2], [1, 1, 3])
assert Solution().minSwaps([1, 2, 3, 4, 5], [1, 2, 3, 4, 5]) == 3 == brute([1, 2, 3, 4, 5], [1, 2, 3, 4, 5])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def lower_bounds_sol(nums, forb):
    bad = Counter(x for x, y in zip(nums, forb) if x == y)
    k = sum(bad.values())
    return (k + 1) // 2, max(bad.values(), default=0)

def sol_dict(nums, forb):
    n = len(nums); cn = {}; cf = {}; bad = {}
    for x in nums: cn[x] = cn.get(x, 0) + 1
    for y in forb: cf[y] = cf.get(y, 0) + 1
    if any(c + cf.get(v, 0) > n for v, c in cn.items()): return -1
    for x, y in zip(nums, forb):
        if x == y: bad[x] = bad.get(x, 0) + 1
    if not bad: return 0
    return max((sum(bad.values()) + 1) // 2, max(bad.values()))

print("Bài 3:", lower_bounds_sol([4, 6, 6, 5], [4, 6, 5, 5]))
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 8); a = [rng.randint(1, 4) for _ in range(n)]; f = [rng.randint(1, 4) for _ in range(n)]
    assert sol_dict(a, f) == Solution().minSwaps(a, f)
print("Bài 4 OK")

Bài 3: (2, 1)
Bài 4 OK


In [24]:
# Bài 5: hai giá trị cấm mỗi vị trí (BFS)
def two_forbidden_bfs_sol(nums, f1, f2):
    n = len(nums); ok = lambda a: all(a[i] != f1[i] and a[i] != f2[i] for i in range(n))
    start = tuple(nums)
    if ok(start): return 0
    seen = {start}; dq = deque([(start, 0)])
    while dq:
        a, d = dq.popleft()
        for i in range(n):
            for j in range(i + 1, n):
                if a[i] == a[j]: continue
                b = list(a); b[i], b[j] = b[j], b[i]; b = tuple(b)
                if b in seen: continue
                if ok(b): return d + 1
                seen.add(b); dq.append((b, d + 1))
    return -1

print("Bài 5 (ví dụ):", two_forbidden_bfs_sol([1, 2, 3, 4], [1, 2, 3, 4], [2, 3, 4, 1]), two_forbidden_bfs_sol([1, 1, 2], [1, 3, 2], [3, 1, 1]))

Bài 5 (ví dụ): 2 -1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cn, cf = Counter(nums), Counter(forbidden)
if any(cn[v] + cf[v] > n for v in cn): return -1
bad = Counter(x for x, y in zip(nums, forbidden) if x == y)
k = sum(bad.values())
return 0 if k == 0 else max((k + 1) // 2, max(bad.values()))
```

**Checklist:**

- [ ] Kiểm tra khả thi trên **toàn mảng** (`c_n(v) + c_f(v) ≤ n`)?
- [ ] `M` là số **xung đột** của giá trị trội?
- [ ] Lấy `max(⌈k/2⌉, M)`?
- [ ] `k = 0` ⇒ `0`?
- [ ] Đã kiểm chứng với BFS trên dữ liệu nhỏ?